# Day 1: Your First SQL Queries

## Objective
Learn the fundamental SQL building blocks: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, and basic functions.

## Table of Contents
1. [Connection Setup](#connection-setup)
2. [SELECT Basics](#select-basics)
3. [WHERE Clause - Filtering Data](#where-clause---filtering-data)
4. [ORDER BY - Sorting Results](#order-by---sorting-results)
5. [LIMIT and OFFSET - Pagination](#limit-and-offset---pagination)
6. [DISTINCT - Removing Duplicates](#distinct---removing-duplicates)
7. [Aliases - Renaming Columns](#aliases---renaming-columns)
8. [String Functions](#string-functions)
9. [Date Functions](#date-functions)
10. [Try It Yourself Exercises](#try-it-yourself-exercises)

## Connection Setup

First, let's create a reusable helper function to run SQL queries. This function connects to our PostgreSQL database, executes a query, returns the results as a pandas DataFrame, and always closes the connection.

**Connection parameters explained:**
- `host="localhost"` — The database is running on your local machine
- `port=5432` — PostgreSQL listens on port 5432 (the default)
- `dbname="week2_db"` — The specific database name
- `user="student"` — Your username
- `password="student123"` — Your password

**Why try/finally?** We use `try/finally` to ensure the database connection is **always closed**, even if the query fails. Leaving connections open wastes resources and can cause problems.

In [1]:
import psycopg2
import pandas as pd

def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL query and return results as a DataFrame."""
    conn = psycopg2.connect(
        host="localhost", port=5432,
        dbname="week2_db", user="student", password="student123"
    )
    try:
        df = pd.read_sql_query(sql, conn)
        return df
    finally:
        conn.close()

print("Helper function defined. Test it with: run_query('SELECT 1 + 1 AS result')")

Helper function defined. Test it with: run_query('SELECT 1 + 1 AS result')


## SELECT Basics

`SELECT` is how you ask the database for data. It's the first word in almost every SQL query.

**The asterisk (`*`) means "all columns"**: `SELECT *` returns every column in the table.

**`LIMIT` prevents accidentally dumping millions of rows.** Always use `LIMIT` when exploring a new table!

The `company.` prefix specifies the schema — like a folder name. It tells PostgreSQL to look for the `employees` table inside the `company` schema.

In [2]:
# Get all columns from the first 10 employees
run_query("SELECT * FROM company.employees LIMIT 10")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,emp_id,first_name,last_name,email,department_id,salary,hire_date,manager_id,is_active
0,1,Alice,Chen,alice.chen@company.com,1,145000.0,2019-03-15,NaN,True
1,2,Bob,Martinez,bob.martinez@company.com,1,125000.0,2020-06-01,1.0,True
2,3,Carol,Johnson,carol.johnson@company.com,1,95000.0,2021-09-10,2.0,True
3,4,David,Kim,david.kim@company.com,1,88000.0,2022-01-15,2.0,True
4,5,Eva,Patel,eva.patel@company.com,1,88000.0,2022-01-15,2.0,True
5,6,Frank,Wilson,frank.wilson@company.com,1,75000.0,2023-04-20,2.0,True
6,7,Grace,Lee,grace.lee@company.com,1,72000.0,2023-07-01,2.0,True
7,8,Hank,Brown,hank.brown@company.com,1,68000.0,2024-02-15,2.0,False
8,9,Iris,Taylor,iris.taylor@company.com,2,115000.0,2018-11-01,NaN,True
9,10,Jack,Anderson,jack.anderson@company.com,2,85000.0,2021-03-15,9.0,True


**Expected output:** A table with 10 rows and 9 columns (emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active). You'll see Alice Chen, Bob Martinez, and the first 8 employees.

In [3]:
# Select only specific columns
run_query("SELECT first_name, salary FROM company.employees LIMIT 10")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,salary
0,Alice,145000.0
1,Bob,125000.0
2,Carol,95000.0
3,David,88000.0
4,Eva,88000.0
5,Frank,75000.0
6,Grace,72000.0
7,Hank,68000.0
8,Iris,115000.0
9,Jack,85000.0


**Expected output:** Two columns (first_name, salary) with 10 rows. Only the data we asked for is returned.

## WHERE Clause - Filtering Data

`WHERE` filters rows based on a condition. Only rows where the condition is TRUE are returned.

Let's explore all the comparison operators one by one.

In [4]:
# = (equals): Find employees in department 1 (Engineering)
run_query("""
    SELECT first_name, last_name, department_id 
    FROM company.employees 
    WHERE department_id = 1
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,department_id
0,Alice,Chen,1
1,Bob,Martinez,1
2,Carol,Johnson,1
3,David,Kim,1
4,Eva,Patel,1
5,Frank,Wilson,1
6,Grace,Lee,1
7,Hank,Brown,1


**Expected output:** All employees with department_id = 1 (Alice Chen, Bob Martinez, Carol Johnson, etc.)

In [5]:
# != or <> (not equals): Employees NOT in department 1
run_query("""
    SELECT first_name, last_name, department_id 
    FROM company.employees 
    WHERE department_id != 1
    LIMIT 10
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,department_id
0,Iris,Taylor,2
1,Jack,Anderson,2
2,Karen,Thomas,2
3,Leo,Jackson,2
4,Maria,White,2
5,Nathan,Harris,3
6,Olivia,Clark,3
7,Peter,Lewis,3
8,Quinn,Walker,3
9,Rachel,Hall,3


**Expected output:** Employees from all departments except Engineering.

In [6]:
# > and < : Employees with salary above 80000
run_query("""
    SELECT first_name, last_name, salary 
    FROM company.employees 
    WHERE salary > 80000
    ORDER BY salary DESC
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,salary
0,Alice,Chen,145000.0
1,Nathan,Harris,135000.0
2,Amy,Adams,130000.0
3,James,Campbell,125000.0
4,Bob,Martinez,125000.0
5,Uma,King,120000.0
6,Iris,Taylor,115000.0
7,Brian,Baker,110000.0
8,Olivia,Clark,105000.0
9,Kelly,Parker,98000.0


**Expected output:** High earners: Alice Chen (145000), Nathan Harris (135000), Amy Adams (130000), etc.

In [7]:
# BETWEEN (inclusive): Salary between 60000 and 90000
run_query("""
    SELECT first_name, last_name, salary 
    FROM company.employees 
    WHERE salary BETWEEN 60000 AND 90000
    ORDER BY salary
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,salary
0,Paula,Morris,60000.0
1,Maria,White,62000.0
2,Zane,Green,65000.0
3,Leo,Jackson,65000.0
4,Gina,Roberts,65000.0
5,Noah,Collins,68000.0
6,Yara,Scott,68000.0
7,Hank,Brown,68000.0
8,Felix,Perez,70000.0
9,Sam,Allen,70000.0


**Expected output:** Mid-range earners. BETWEEN includes both endpoints (60000 and 90000).

In [8]:
# IN: Employees in departments 1, 3, or 5
run_query("""
    SELECT first_name, last_name, department_id 
    FROM company.employees 
    WHERE department_id IN (1, 3, 5)
    ORDER BY department_id, last_name
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,department_id
0,Hank,Brown,1
1,Alice,Chen,1
2,Carol,Johnson,1
3,David,Kim,1
4,Grace,Lee,1
5,Bob,Martinez,1
6,Eva,Patel,1
7,Frank,Wilson,1
8,Sam,Allen,3
9,Olivia,Clark,3


**Expected output:** Employees from Engineering (1), Finance (3), and Sales (5) departments.

In [9]:
# LIKE: Pattern matching with strings
# % = any characters, _ = exactly one character
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    WHERE first_name LIKE 'A%'  -- Names starting with 'A'
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name
0,Alice,Chen
1,Amy,Adams


**Expected output:** Alice, Amy (all names starting with A).

In [10]:
# LIKE with % in the middle: names containing 'an'
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    WHERE first_name LIKE '%an%'
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name
0,Frank,Wilson
1,Hank,Brown
2,Nathan,Harris
3,Zane,Green
4,Brian,Baker


**Expected output:** Names like Nathan, etc. — any name with 'an' in it.

### IMPORTANT: IS NULL vs = NULL

**Common beginner trap:** You cannot use `= NULL` to find NULL values. NULL means "unknown," and "unknown = unknown" is not TRUE — it's also unknown!

Always use `IS NULL` or `IS NOT NULL`.

In [11]:
# IS NULL: Employees with no manager (top-level managers)
run_query("""
    SELECT first_name, last_name, manager_id 
    FROM company.employees 
    WHERE manager_id IS NULL
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,manager_id
0,Alice,Chen,None
1,Iris,Taylor,None
2,Nathan,Harris,None
3,Uma,King,None
4,Amy,Adams,None
5,James,Campbell,None
6,Paula,Morris,None


**Expected output:** Alice Chen, Iris Taylor, Nathan Harris, Uma King, Amy Adams, James Campbell — the directors with no managers above them.

In [12]:
# WRONG WAY (demonstrating the error!):
# This returns NO rows because = NULL always evaluates to NULL (unknown)
run_query("""
    SELECT first_name, last_name, manager_id 
    FROM company.employees 
    WHERE manager_id = NULL
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,manager_id


**Expected output:** Empty table! See? `= NULL` never matches. Always use `IS NULL`.

## ORDER BY - Sorting Results

`ORDER BY` sorts the result set. Default is ascending (`ASC`). Use `DESC` for descending.

In [13]:
# Sort by salary descending (highest first)
run_query("""
    SELECT first_name, last_name, salary 
    FROM company.employees 
    ORDER BY salary DESC
    LIMIT 10
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,salary
0,Alice,Chen,145000.0
1,Nathan,Harris,135000.0
2,Amy,Adams,130000.0
3,Bob,Martinez,125000.0
4,James,Campbell,125000.0
5,Uma,King,120000.0
6,Iris,Taylor,115000.0
7,Brian,Baker,110000.0
8,Olivia,Clark,105000.0
9,Kelly,Parker,98000.0


**Expected output:** Top 10 earners starting with Alice Chen (145000).

In [14]:
# Sort by department, then by salary within each department
run_query("""
    SELECT first_name, last_name, department_id, salary 
    FROM company.employees 
    ORDER BY department_id ASC, salary DESC
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,department_id,salary
0,Alice,Chen,1.0,145000.0
1,Bob,Martinez,1.0,125000.0
2,Carol,Johnson,1.0,95000.0
3,David,Kim,1.0,88000.0
4,Eva,Patel,1.0,88000.0
5,Frank,Wilson,1.0,75000.0
6,Grace,Lee,1.0,72000.0
7,Hank,Brown,1.0,68000.0
8,Iris,Taylor,2.0,115000.0
9,Jack,Anderson,2.0,85000.0


**Expected output:** All employees grouped by department (1, 2, 3, 4, 5, 6), with highest-paid in each department first.

## LIMIT and OFFSET - Pagination

`LIMIT` controls how many rows to return. `OFFSET` controls how many rows to skip.

This is the pattern web apps use for pagination: "Page 3" means `LIMIT 10 OFFSET 20` (skip 20, show next 10).

In [15]:
# Page 1: First 10 employees
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    ORDER BY emp_id
    LIMIT 10 OFFSET 0
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name
0,Alice,Chen
1,Bob,Martinez
2,Carol,Johnson
3,David,Kim
4,Eva,Patel
5,Frank,Wilson
6,Grace,Lee
7,Hank,Brown
8,Iris,Taylor
9,Jack,Anderson


**Expected output:** Alice Chen through Iris Taylor (emp_id 1-10).

In [16]:
# Page 3: Skip 20, get the next 10
run_query("""
    SELECT first_name, last_name 
    FROM company.employees 
    ORDER BY emp_id
    LIMIT 10 OFFSET 20
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name
0,Uma,King
1,Victor,Wright
2,Wendy,Lopez
3,Xavier,Hill
4,Yara,Scott
5,Zane,Green
6,Amy,Adams
7,Brian,Baker
8,Cindy,Nelson
9,Derek,Carter


**Expected output:** Uma King through Henry Turner (emp_id 21-30).

## DISTINCT - Removing Duplicates

`DISTINCT` removes duplicate rows. Only unique combinations are returned.

In [17]:
# Get unique department IDs (without seeing all employees)
run_query("""
    SELECT DISTINCT department_id 
    FROM company.employees 
    ORDER BY department_id
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,department_id
0,1.0
1,2.0
2,3.0
3,4.0
4,5.0
5,6.0
6,NaN


**Expected output:** Unique department IDs: 1, 2, 3, 4, 5, 6, and NULL (for Paula Morris who has no department).

## Aliases - Renaming Columns

`AS` renames a column in the output. This is especially useful for computed columns.

In [18]:
# Rename columns and compute annual salary
run_query("""
    SELECT 
        first_name AS name, 
        salary * 12 AS annual_salary
    FROM company.employees 
    LIMIT 10
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,name,annual_salary
0,Alice,1740000.0
1,Bob,1500000.0
2,Carol,1140000.0
3,David,1056000.0
4,Eva,1056000.0
5,Frank,900000.0
6,Grace,864000.0
7,Hank,816000.0
8,Iris,1380000.0
9,Jack,1020000.0


**Expected output:** Columns named "name" and "annual_salary" (monthly salary × 12).

## String Functions

PostgreSQL has built-in string functions: `UPPER()`, `LOWER()`, `LENGTH()`, `CONCAT()`, and `SUBSTRING()`.

In [19]:
# UPPER and LOWER
run_query("""
    SELECT 
        first_name,
        UPPER(first_name) AS upper_name,
        LOWER(first_name) AS lower_name
    FROM company.employees 
    LIMIT 5
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,upper_name,lower_name
0,Alice,ALICE,alice
1,Bob,BOB,bob
2,Carol,CAROL,carol
3,David,DAVID,david
4,Eva,EVA,eva


**Expected output:** first_name alongside uppercase and lowercase versions.

In [20]:
# LENGTH and CONCAT
run_query("""
    SELECT 
        first_name,
        LENGTH(first_name) AS name_length,
        CONCAT(first_name, ' ', last_name) AS full_name
    FROM company.employees 
    LIMIT 5
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,name_length,full_name
0,Alice,5,Alice Chen
1,Bob,3,Bob Martinez
2,Carol,5,Carol Johnson
3,David,5,David Kim
4,Eva,3,Eva Patel


**Expected output:** Name length and full name (first + last with space in between).

## Date Functions

PostgreSQL provides powerful date/time functions: `EXTRACT()`, `AGE()`, `NOW()`, and `DATE_TRUNC()`.

In [21]:
# EXTRACT year from hire_date
run_query("""
    SELECT 
        first_name,
        hire_date,
        EXTRACT(YEAR FROM hire_date) AS hire_year
    FROM company.employees 
    LIMIT 10
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,hire_date,hire_year
0,Alice,2019-03-15,2019.0
1,Bob,2020-06-01,2020.0
2,Carol,2021-09-10,2021.0
3,David,2022-01-15,2022.0
4,Eva,2022-01-15,2022.0
5,Frank,2023-04-20,2023.0
6,Grace,2023-07-01,2023.0
7,Hank,2024-02-15,2024.0
8,Iris,2018-11-01,2018.0
9,Jack,2021-03-15,2021.0


**Expected output:** Hire dates alongside the year they were hired.

In [22]:
# AGE: How long has each employee been with the company?
run_query("""
    SELECT 
        first_name,
        hire_date,
        AGE(hire_date) AS tenure
    FROM company.employees 
    ORDER BY hire_date
    LIMIT 10
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,hire_date,tenure
0,Nathan,2017-05-20,3350 days
1,Amy,2018-07-01,2944 days
2,Iris,2018-11-01,2819 days
3,Uma,2019-01-10,2750 days
4,Alice,2019-03-15,2685 days
5,James,2019-08-15,2530 days
6,Brian,2020-02-15,2350 days
7,Bob,2020-06-01,2244 days
8,Olivia,2020-09-15,2135 days
9,Victor,2020-11-20,2070 days


**Expected output:** Shows how many years, months, and days each employee has been with the company.

In [23]:
# DATE_TRUNC: Round a date to the first day of the month
run_query("""
    SELECT 
        hire_date,
        DATE_TRUNC('month', hire_date) AS month_start
    FROM company.employees 
    LIMIT 10
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,hire_date,month_start
0,2019-03-15,2019-03-01 00:00:00+00:00
1,2020-06-01,2020-06-01 00:00:00+00:00
2,2021-09-10,2021-09-01 00:00:00+00:00
3,2022-01-15,2022-01-01 00:00:00+00:00
4,2022-01-15,2022-01-01 00:00:00+00:00
5,2023-04-20,2023-04-01 00:00:00+00:00
6,2023-07-01,2023-07-01 00:00:00+00:00
7,2024-02-15,2024-02-01 00:00:00+00:00
8,2018-11-01,2018-11-01 00:00:00+00:00
9,2021-03-15,2021-03-01 00:00:00+00:00


**Expected output:** hire_date and the same date rounded to the first of that month (e.g., 2019-03-15 → 2019-03-01).

## Try It Yourself Exercises

Now it's your turn! Try these exercises. Each one has a hidden solution you can reveal after attempting it yourself.

### Exercise 1
Find all **active** employees hired **after 2023-06-01**, ordered by hire date (most recent first).

<details>
<summary><b>🔍 Click to reveal solution</b></summary>

```sql
SELECT first_name, last_name, hire_date, is_active
FROM company.employees
WHERE is_active = TRUE 
  AND hire_date > '2023-06-01'
ORDER BY hire_date DESC;
```

**Expected output:** Recent hires starting with Paula Morris (2024-07-01), Oscar Stewart, Henry Turner, etc.
</details>

In [24]:
# Your solution here:
run_query("""
    SELECT 
        first_name, last_name, hire_date, is_active
    FROM company.employees
    Where is_active = TRUE
        And hire_date >= '2023-06-01'
    ORDER BY hire_date DESC;
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,last_name,hire_date,is_active
0,Paula,Morris,2024-07-01,True
1,Oscar,Stewart,2024-05-01,True
2,Henry,Turner,2024-04-01,True
3,Gina,Roberts,2023-11-01,True
4,Noah,Collins,2023-10-15,True
5,Sam,Allen,2023-08-15,True
6,Grace,Lee,2023-07-01,True
7,Leo,Jackson,2023-06-01,True


### Exercise 2
Find employees whose **last name contains 'son'** (like Johnson, Nelson), showing their full name and salary, ordered by salary descending.

<details>
<summary><b>🔍 Click to reveal solution</b></summary>

```sql
SELECT 
    CONCAT(first_name, ' ', last_name) AS full_name,
    salary
FROM company.employees
WHERE last_name LIKE '%son%'
ORDER BY salary DESC;
```

**Expected output:** Carol Johnson, Cindy Nelson, Eva Patels, etc. with their salaries.
</details>

In [26]:
# Your solution here:
run_query("""
    SELECT 
        CONCAT(first_name, ' ', last_name) AS full_name,
        salary
    FROM company.employees
    WHERE last_name LIKE '%son%'
    ORDER BY salary DESC;  
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,full_name,salary
0,Carol Johnson,95000.0
1,Cindy Nelson,88000.0
2,Jack Anderson,85000.0
3,Frank Wilson,75000.0
4,Leo Jackson,65000.0


### Exercise 3
Find the **email addresses** of employees who have **no manager** (manager_id IS NULL), showing their first name and email.

<details>
<summary><b>🔍 Click to reveal solution</b></summary>

```sql
SELECT first_name, email
FROM company.employees
WHERE manager_id IS NULL;
```

**Expected output:** The 6 department directors: Alice Chen, Iris Taylor, Nathan Harris, Uma King, Amy Adams, James Campbell.
</details>

In [27]:
# Your solution here:
run_query("""
    SELECT 
        first_name, email
    From company.employees
    Where manager_id IS NULL;
    
""")

/tmp/ipykernel_12675/390252506.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql, conn)


,first_name,email
0,Alice,alice.chen@company.com
1,Iris,iris.taylor@company.com
2,Nathan,nathan.harris@company.com
3,Uma,uma.king@company.com
4,Amy,amy.adams@company.com
5,James,james.campbell@company.com
6,Paula,paula.morris@company.com
